# S03 — SQL Advanced Patterns

**Topics:** Recursive CTEs, PIVOT/UNPIVOT, set operations (UNION/INTERSECT/EXCEPT), advanced aggregation (GROUPING SETS, CUBE, ROLLUP), query optimization concepts, DuckDB-specific features (ASOF JOIN, list aggregation, JSON).

**Reference:** [DuckDB docs](https://duckdb.org/docs/)

These are the patterns that separate intermediate from advanced SQL practitioners — and the ones that come up in senior DS interviews.


In [ ]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml

retail_raw = fetch_openml(name='onlineretail', version=1, as_frame=True, parser='auto').frame
retail = retail_raw.copy()
retail.columns = [c.strip() for c in retail.columns]
retail['InvoiceDate'] = pd.to_datetime(retail['InvoiceDate'])
retail['Quantity'] = pd.to_numeric(retail['Quantity'], errors='coerce')
retail['UnitPrice'] = pd.to_numeric(retail['UnitPrice'], errors='coerce')
retail['CustomerID'] = pd.to_numeric(retail['CustomerID'], errors='coerce')
retail['Revenue'] = retail['Quantity'] * retail['UnitPrice']
retail['Month'] = retail['InvoiceDate'].dt.to_period('M').astype(str)
retail = retail[retail['Quantity'] > 0].dropna(subset=['CustomerID'])
retail['CustomerID'] = retail['CustomerID'].astype(int)

# Synthetic org hierarchy for recursive CTE
np.random.seed(42)
org = pd.DataFrame({
    'employee_id': range(1, 21),
    'name': [f'Employee_{i}' for i in range(1, 21)],
    'manager_id': [None, 1, 1, 1, 2, 2, 3, 3, 4, 4,
                   5, 5, 6, 7, 8, 9, 10, 11, 12, 13],
    'department': np.random.choice(['Sales','Tech','Ops','Finance'], 20),
    'salary': np.random.randint(50000, 150000, 20)
})

con = duckdb.connect()
con.register('retail', retail)
con.register('org', org)

def q(sql): return con.execute(sql).df()
print("Ready. Tables: retail, org")
q("SELECT * FROM org LIMIT 5")

---
## Exercise 1 — Recursive CTE: Org Hierarchy

**Business question:** Starting from the CEO (employee_id=1), traverse the entire org hierarchy and return every employee with their `level` (CEO=1, direct reports=2, etc.) and their full management chain as a string like `'Employee_1 > Employee_2 > Employee_5'`.

**Concept:** Recursive CTEs have two parts:
1. **Anchor member** — the base case (e.g. the CEO)
2. **Recursive member** — joins the CTE back to itself to go one level deeper

```sql
WITH RECURSIVE hierarchy AS (
    -- Anchor: starting point
    SELECT ...
    UNION ALL
    -- Recursive: join to self
    SELECT ... FROM org JOIN hierarchy ON ...
)
SELECT * FROM hierarchy
```

In [ ]:
sql1 = """
-- YOUR SQL HERE
"""

result1 = q(sql1)
result1.sort_values('level')

In [ ]:
# --- ASSERTIONS ---
assert len(result1) == len(org), f"All 20 employees must appear"
level_col = [c for c in result1.columns if 'level' in c.lower() or 'depth' in c.lower()][0]
assert result1[level_col].min() == 1, "CEO must be level 1"
assert result1[level_col].max() >= 3, "Tree must have at least 3 levels"
chain_col = [c for c in result1.columns if 'chain' in c.lower() or 'path' in c.lower()]
if chain_col:
    ceo_chain = result1.loc[result1[level_col]==1, chain_col[0]].iloc[0]
    assert 'Employee_1' in str(ceo_chain)
print(f"✓ Exercise 1 passed — {result1[level_col].max()} levels deep")
print(result1.sort_values(level_col).to_string(index=False))

---
## Exercise 2 — GROUPING SETS, ROLLUP, CUBE

**Business question:** Build a multi-dimensional revenue summary with subtotals — the kind used in management reporting.

Use `GROUPING SETS` to compute revenue for:
- Each (Country, Month) combination
- Each Country subtotal (all months)
- Each Month subtotal (all countries)
- Grand total

Add a `row_type` column: `'Detail'`, `'Country Subtotal'`, `'Month Subtotal'`, or `'Grand Total'` using `GROUPING()` function.

Limit to top 5 countries by revenue. Sort logically: grand total last.

In [ ]:
top5 = q("SELECT Country FROM retail GROUP BY Country ORDER BY SUM(Revenue) DESC LIMIT 5")['Country'].tolist()
con.execute(f"CREATE OR REPLACE TABLE retail_top5 AS SELECT * FROM retail WHERE Country IN {tuple(top5)}")

sql2 = """
-- YOUR SQL HERE
-- Use GROUPING SETS ((Country, Month), (Country), (Month), ())
-- Use GROUPING(Country) and GROUPING(Month) to identify row type
"""

result2 = q(sql2)
result2.tail(10)

In [ ]:
# --- ASSERTIONS ---
row_type_col = [c for c in result2.columns if 'type' in c.lower() or 'row' in c.lower()][0]
assert 'Grand Total' in result2[row_type_col].values
assert 'Country Subtotal' in result2[row_type_col].values
assert 'Month Subtotal' in result2[row_type_col].values
assert 'Detail' in result2[row_type_col].values
# Grand total revenue should equal sum of all detail rows
rev_col = [c for c in result2.columns if 'rev' in c.lower()][0]
grand = float(result2.loc[result2[row_type_col]=='Grand Total', rev_col].iloc[0])
details = float(result2.loc[result2[row_type_col]=='Detail', rev_col].sum())
assert abs(grand - details) < 1, "Grand total must match sum of details"
print(f"✓ Exercise 2 passed — {len(result2)} rows with subtotals")

---
## Exercise 3 — PIVOT and UNPIVOT

**Business question:** Reshape the monthly data into a wide format where each row is a country and each column is a month's revenue — then unpivot it back.

**Part A:** Use DuckDB's `PIVOT` to create a wide table: rows = Country (top 6), columns = months (each month a separate column), values = total revenue. Fill NULLs with 0.

**Part B:** Use `UNPIVOT` to convert the wide table back to long format with columns: Country, Month, Revenue.

In [ ]:
sql3a = """
-- YOUR SQL HERE — PIVOT
-- Hint: PIVOT table ON column USING aggregation
"""

result3a = q(sql3a)
print(f"Wide shape: {result3a.shape}")
result3a

In [ ]:
# Register the wide result for unpivot
con.register('wide_revenue', result3a)

sql3b = """
-- YOUR SQL HERE — UNPIVOT
-- Hint: UNPIVOT wide_revenue ON (month_columns) INTO NAME Month VALUE Revenue
"""

result3b = q(sql3b)
print(f"Long shape: {result3b.shape}")
result3b.head()

In [ ]:
# --- ASSERTIONS ---
assert result3a.shape[0] == 6, "Wide table must have 6 country rows"
assert result3a.shape[1] > 6, "Wide table must have month columns"
assert result3b.shape[1] >= 3, "Long table must have Country, Month, Revenue"
assert len(result3b) == result3a.shape[0] * (result3a.shape[1] - 1), \
    "Long rows = wide rows × month columns"
print(f"✓ Exercise 3 passed — Pivot: {result3a.shape}, Unpivot: {result3b.shape}")

---
## Exercise 4 — Set Operations: UNION, INTERSECT, EXCEPT

**Business question:** Analyze customer overlap between time periods.

1. Find customers who bought in **both** the first half and second half of the dataset (`INTERSECT`)
2. Find customers who bought in the first half but **not** the second half — churned (`EXCEPT`)
3. Find customers who bought in the second half but not the first — new (`EXCEPT` reversed)
4. `UNION ALL` results 2 and 3 into a single table with a `status` column: `'Churned'` or `'New'`

Return summary counts by status.

In [ ]:
midpoint = q("SELECT MEDIAN(InvoiceDate) as mid FROM retail")['mid'].iloc[0]
print(f"Dataset midpoint: {midpoint}")

sql4 = """
-- YOUR SQL HERE
-- Step 1: Define h1_customers (before midpoint) and h2_customers (after)
-- Step 2: Use INTERSECT, EXCEPT for each group
-- Step 3: UNION ALL churned and new with status label
-- Step 4: Return COUNT by status
"""

result4 = q(sql4)
result4

In [ ]:
# --- ASSERTIONS ---
status_col = [c for c in result4.columns if 'status' in c.lower() or result4[c].dtype == object][0]
assert set(result4[status_col]) == {'Churned', 'New'}
count_col = [c for c in result4.columns if 'count' in c.lower() or result4[c].dtype in [np.int64,'int64']][0]
assert (result4[count_col] > 0).all()
print(f"✓ Exercise 4 passed")
print(result4.to_string(index=False))

---
## Exercise 5 — LIST Aggregation & Array Functions

**Business question:** For each customer, aggregate their purchase history into a structured array.

Use DuckDB's `LIST` aggregation to compute:
- `product_list`: list of distinct StockCodes purchased (sorted)
- `n_unique_products`: count of unique products
- `monthly_revenues`: list of monthly revenue totals ordered by month
- `best_month_revenue`: MAX from the monthly_revenues list
- `has_repeat_product`: TRUE if any product appears more than once across invoices

Return top 20 customers by total revenue.

In [ ]:
sql5 = """
-- YOUR SQL HERE
-- DuckDB list functions: LIST_AGG, ARRAY_AGG, LIST_DISTINCT, LIST_SORT, ARRAY_LENGTH
"""

result5 = q(sql5)
result5.head()

In [ ]:
# --- ASSERTIONS ---
assert len(result5) == 20
for col in ['product_list','n_unique_products','monthly_revenues']:
    assert col in result5.columns, f"Missing: {col}"
assert (result5['n_unique_products'] > 0).all()
print("✓ Exercise 5 passed")
print(result5[['CustomerID','n_unique_products','best_month_revenue']].head())

---
## Exercise 6 — ASOF JOIN: Time-Series Alignment

**Business question:** Match each transaction to the most recent exchange rate at the time of the transaction. ASOF JOIN finds the nearest preceding record — essential for time-series data alignment.

First, create synthetic GBP→EUR exchange rate history, then use ASOF JOIN to attach the rate valid at each invoice date.

In [ ]:
# Synthetic FX rates — one rate change per week
date_range = pd.date_range('2010-12-01', '2011-12-31', freq='W')
np.random.seed(7)
fx_rates = pd.DataFrame({
    'effective_date': date_range,
    'gbp_eur_rate': np.cumprod(1 + np.random.normal(0.0002, 0.008, len(date_range))) * 1.17
})
con.register('fx_rates', fx_rates)

sql6 = """
-- YOUR SQL HERE
-- Use ASOF JOIN retail r ON fx_rates f WHERE f.effective_date <= r.InvoiceDate
-- Compute revenue_eur = Revenue * gbp_eur_rate
-- Return: InvoiceNo, InvoiceDate, Revenue, gbp_eur_rate, revenue_eur
-- LIMIT 1000 for performance
"""

result6 = q(sql6)
result6.head()

In [ ]:
# --- ASSERTIONS ---
assert 'gbp_eur_rate' in result6.columns
assert 'revenue_eur' in result6.columns
assert (result6['gbp_eur_rate'] > 1.0).all(), "GBP→EUR rate must be > 1"
computed = result6['Revenue'] * result6['gbp_eur_rate']
assert np.allclose(result6['revenue_eur'], computed, rtol=1e-3)
print(f"✓ Exercise 6 passed — FX rates applied to {len(result6)} transactions")
print(f"Rate range: {result6['gbp_eur_rate'].min():.4f} – {result6['gbp_eur_rate'].max():.4f}")

---
## Exercise 7 — Query Optimization: EXPLAIN & Indexes

**Concept:** Understanding query execution plans is a senior DS skill. DuckDB's `EXPLAIN` shows you how a query will run.

1. Write a slow version of the customer health query (no optimization — full scan, cartesian).
2. Run `EXPLAIN` on it and capture the plan.
3. Write an optimized version using:
   - Pre-aggregation in CTEs before joining
   - Filtering early (push predicates down)
   - Avoiding repeated subqueries
4. Compare row counts at each step.
5. Document what optimization you applied in the markdown cell.

In [ ]:
# Slow version — intentionally naive
sql7_slow = """
SELECT
    r.CustomerID,
    SUM(r.Revenue) as total_revenue
FROM retail r
WHERE r.CustomerID IN (
    SELECT CustomerID FROM retail WHERE Revenue > 100
)
GROUP BY r.CustomerID
HAVING SUM(r.Revenue) > 500
"""

# Run EXPLAIN on the slow version
explain_result = con.execute(f"EXPLAIN {sql7_slow}").df()
print("=== QUERY PLAN (slow version) ===")
print(explain_result.to_string(index=False))

In [ ]:
sql7_optimized = """
-- YOUR OPTIMIZED VERSION HERE
-- Same result, but push filters earlier and avoid the correlated subquery
"""

result7_slow = q(sql7_slow)
result7_opt = q(sql7_optimized)

In [ ]:
# --- ASSERTIONS ---
# Results must be identical
slow_sorted = result7_slow.sort_values('CustomerID').reset_index(drop=True)
opt_sorted = result7_opt.sort_values('CustomerID').reset_index(drop=True)
assert len(slow_sorted) == len(opt_sorted), "Optimized query must return same row count"
assert np.allclose(
    slow_sorted['total_revenue'].values,
    opt_sorted['total_revenue'].values,
    rtol=1e-3
), "Revenue values must match"
print(f"✓ Exercise 7 passed — {len(result7_opt)} customers, results match")

**Optimization applied:** *(What did you change and why is it faster? Reference the EXPLAIN output.)*

---
## Exercise 8 — Reading & Writing Parquet

**Business context:** In production, data lives in files — not just DataFrames. DuckDB reads Parquet directly, without loading into memory first. This is how modern lakehouses work.

1. Write the retail DataFrame to a Parquet file at `/tmp/retail.parquet`
2. Use DuckDB to query it **directly from file** (not via registered DataFrame)
3. Compute monthly revenue summary from the Parquet file
4. Write the result to `/tmp/monthly_summary.parquet`
5. Read it back and verify

In [ ]:
# Write to parquet
retail.to_parquet('/tmp/retail.parquet', index=False)
print("Written: /tmp/retail.parquet")

# YOUR SQL HERE — query directly from file, write result
sql8_read = """
-- Read from file and aggregate
-- SELECT ... FROM read_parquet('/tmp/retail.parquet') ...
"""

sql8_write = """
-- Write result to parquet
-- COPY (...) TO '/tmp/monthly_summary.parquet' (FORMAT PARQUET)
"""

result8 = q(sql8_read)
con.execute(sql8_write)

# Read back
result8_verify = q("SELECT * FROM read_parquet('/tmp/monthly_summary.parquet')")
result8_verify

In [ ]:
# --- ASSERTIONS ---
import os
assert os.path.exists('/tmp/monthly_summary.parquet')
assert len(result8_verify) > 0
assert len(result8_verify) == len(result8), "Written and read-back must match"
print(f"✓ Exercise 8 passed — {len(result8_verify)} months written and read back")

---
## Exercise 9 — Macro & Reusable SQL Functions

**Concept:** DuckDB supports SQL macros — reusable query fragments. This is the SQL equivalent of writing a Python function.

1. Create a macro `revenue_summary(tbl, grp_col)` that computes total_revenue, n_orders, n_customers for any group column.
2. Create a scalar macro `tier_label(revenue)` that returns `'Low'`/`'Mid'`/`'High'`/`'Premium'` based on revenue thresholds.
3. Use both macros in a final query.

In [ ]:
# Create macros
con.execute("""
-- YOUR MACRO DEFINITIONS HERE
-- CREATE OR REPLACE MACRO tier_label(revenue) AS ...
""")

sql9 = """
-- YOUR SQL HERE using the macros
SELECT
    Country,
    SUM(Revenue) as total_revenue,
    tier_label(SUM(Revenue)) as tier
FROM retail
GROUP BY Country
ORDER BY total_revenue DESC
LIMIT 10
"""

result9 = q(sql9)
result9

In [ ]:
# --- ASSERTIONS ---
assert 'tier' in result9.columns
assert set(result9['tier']).issubset({'Low','Mid','High','Premium'})
assert result9['total_revenue'].is_monotonic_decreasing
print("✓ Exercise 9 passed")
print(result9.to_string(index=False))

---
## Exercise 10 — Capstone: Full Data Pipeline in SQL

**Spec:** Build a complete data pipeline entirely in DuckDB SQL — from raw retail data to a final analytics table saved as Parquet.

Pipeline steps (each a CTE or VIEW):
1. **Clean**: filter out negative quantities, nulls, zero prices
2. **Enrich**: add month, quarter, day_of_week, revenue_tier
3. **Customer RFM**: compute R, F, M scores
4. **Segment**: assign segment names based on RFM combination
5. **Country metrics**: revenue, orders, customers per country with growth vs prior month
6. **Final join**: combine customer + country metrics into one wide table
7. **Write** final table to `/tmp/analytics_output.parquet`
8. **Validate**: read back and assert row counts, column presence, no nulls in key columns

In [ ]:
sql10 = """
-- YOUR FULL PIPELINE SQL HERE
-- 6 CTEs minimum, ends with COPY ... TO parquet
"""

# Execute pipeline
con.execute(sql10)

# Read back and validate
final = q("SELECT * FROM read_parquet('/tmp/analytics_output.parquet')")
print(f"Output shape: {final.shape}")
final.head()

In [ ]:
# --- ASSERTIONS ---
import os
assert os.path.exists('/tmp/analytics_output.parquet'), "Output file must exist"
assert len(final) > 0, "Output must have rows"
assert final.shape[1] >= 8, "Output must have at least 8 columns"
# Key columns present
cols_lower = final.columns.str.lower().tolist()
for key in ['customerid', 'revenue', 'segment']:
    assert any(key in c for c in cols_lower), f"Missing key column: {key}"
# No nulls in CustomerID
cust_col = [c for c in final.columns if 'customer' in c.lower()][0]
assert final[cust_col].isna().sum() == 0, "CustomerID must not have nulls"
print(f"✓ Exercise 10 passed — Pipeline output: {final.shape[0]} rows, {final.shape[1]} columns")
print(final.dtypes)